# Neural Search: Dense Passage Retrieval and Modern Retrieval

## Learning Objectives
1. Understand dense embeddings and semantic similarity for retrieval
2. Implement bi-encoder architecture for query-passage encoding
3. Build large-scale vector indices using FAISS for efficient retrieval
4. Evaluate and optimize neural retrievers on open-domain QA tasks

## Cell 2: Imports and Device Setup

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from typing import List, Tuple, Dict, Optional
import time
import matplotlib.pyplot as plt

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

## Level 1: Basic Dense Retrieval with Cosine Similarity

In [ ]:
class SimpleDenseRetriever:
    """Simple dense retriever: encode documents, search by cosine similarity."""
    
    def __init__(self, embedding_dim: int = 768):
        """Initialize retriever.
        
        Args:
            embedding_dim: Dimension of embeddings
        """
        self.embedding_dim = embedding_dim
        self.passages = []
        self.embeddings = None  # (num_passages, embedding_dim)
    
    def encode(self, texts: List[str]) -> torch.Tensor:
        """Generate embeddings for texts (simplified: random for demo).
        
        In production, use: from sentence_transformers import SentenceTransformer
        
        Args:
            texts: List of text strings
        
        Returns:
            Embeddings tensor (len(texts), embedding_dim)
        """
        # For demo: hash-based embeddings (deterministic, not learned)
        embs = []
        for text in texts:
            # Simple hash-based embedding: use text hash as seed
            np.random.seed(hash(text) % (2**31))
            emb = np.random.randn(self.embedding_dim).astype(np.float32)
            # L2 normalize
            emb = emb / (np.linalg.norm(emb) + 1e-8)
            embs.append(emb)
        
        return torch.from_numpy(np.array(embs))
    
    def index_passages(self, passages: List[str]):
        """Encode and store all passages.
        
        Args:
            passages: List of passage texts
        """
        self.passages = passages
        self.embeddings = self.encode(passages)  # (num_passages, dim)
        print(f"Indexed {len(passages)} passages, embedding dim: {self.embedding_dim}")
    
    def retrieve(self, query: str, top_k: int = 3) -> Tuple[List[str], List[float]]:
        """Retrieve top-K passages for a query.
        
        Args:
            query: Query text
            top_k: Number of passages to retrieve
        
        Returns:
            (passages, scores) - top-K passages and similarity scores
        """
        if self.embeddings is None:
            raise ValueError("Call index_passages first")
        
        # Encode query
        q_emb = self.encode([query])[0]  # (dim,)
        
        # Cosine similarity: q_emb^T * p_embs (after L2 norm, this is dot product)
        scores = torch.matmul(self.embeddings, q_emb)  # (num_passages,)
        
        # Top-K
        top_k_scores, top_k_idx = torch.topk(scores, min(top_k, len(self.passages)))
        
        retrieved = [self.passages[i] for i in top_k_idx.numpy()]
        retrieved_scores = top_k_scores.numpy().tolist()
        
        return retrieved, retrieved_scores

# Test basic retriever
print("=== Basic Dense Retrieval ===")
retriever = SimpleDenseRetriever(embedding_dim=64)

corpus = [
    "Paris is the capital of France.",
    "The Eiffel Tower is in Paris and is 330 meters tall.",
    "France is a country in Western Europe.",
    "Tokyo is the capital and largest city of Japan.",
    "Japan is an island nation in East Asia.",
    "Mount Fuji is the highest mountain in Japan.",
]

retriever.index_passages(corpus)

# Test queries
queries = [
    "What is the capital of France?",
    "How tall is Mount Fuji?",
]

for query in queries:
    passages, scores = retriever.retrieve(query, top_k=2)
    print(f"\nQuery: {query}")
    for i, (p, s) in enumerate(zip(passages, scores), 1):
        print(f"  [{i}] (score={s:.3f}) {p[:50]}...")

## Level 2: Bi-Encoder with Contrastive Training

In [ ]:
class BiEncoderRetriever(torch.nn.Module):
    """Bi-encoder: separate encoders for queries and passages.
    
    Trained with contrastive loss to pull relevant pairs together.
    """
    
    def __init__(self, embedding_dim: int = 256, hidden_dim: int = 128):
        super().__init__()
        self.embedding_dim = embedding_dim
        
        # Query encoder: simple MLP
        self.query_encoder = torch.nn.Sequential(
            torch.nn.Linear(50, hidden_dim),  # Input: 50-dim text hash
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, embedding_dim),
        )
        
        # Passage encoder: same architecture (could be separate)
        self.passage_encoder = torch.nn.Sequential(
            torch.nn.Linear(50, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, embedding_dim),
        )
    
    def text_to_features(self, text: str) -> torch.Tensor:
        """Convert text to fixed-size feature vector (simplified)."""
        np.random.seed(hash(text) % (2**31))
        features = np.random.randn(50).astype(np.float32)
        return torch.from_numpy(features)
    
    def encode_queries(self, queries: List[str]) -> torch.Tensor:
        """Encode queries.
        
        Args:
            queries: List of query texts
        
        Returns:
            Query embeddings (batch_size, embedding_dim)
        """
        features = torch.stack([self.text_to_features(q) for q in queries])
        embeddings = self.query_encoder(features)
        # L2 normalize
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return embeddings
    
    def encode_passages(self, passages: List[str]) -> torch.Tensor:
        """Encode passages.
        
        Args:
            passages: List of passage texts
        
        Returns:
            Passage embeddings (batch_size, embedding_dim)
        """
        features = torch.stack([self.text_to_features(p) for p in passages])
        embeddings = self.passage_encoder(features)
        # L2 normalize
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return embeddings
    
    def forward(self, queries: List[str], passages: List[str]) -> torch.Tensor:
        """Compute similarity scores between queries and passages.
        
        Args:
            queries: List of queries
            passages: List of passages
        
        Returns:
            Similarity matrix (len(queries), len(passages))
        """
        q_emb = self.encode_queries(queries)  # (B, D)
        p_emb = self.encode_passages(passages)  # (N, D)
        # Dot product (cosine after L2 norm)
        scores = torch.matmul(q_emb, p_emb.t())  # (B, N)
        return scores

class DPRTrainer:
    """Train bi-encoder with contrastive loss."""
    
    def __init__(self, embedding_dim: int = 256):
        self.model = BiEncoderRetriever(embedding_dim=embedding_dim)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-3)
        self.temperatures = []  # For tracking
    
    def compute_contrastive_loss(self, scores: torch.Tensor) -> torch.Tensor:
        """Compute in-batch contrastive loss (all negatives from batch).
        
        Args:
            scores: Similarity scores (batch_size, batch_size)
                   Diagonal are positives, off-diagonal are negatives
        
        Returns:
            Scalar loss
        """
        batch_size = scores.shape[0]
        labels = torch.arange(batch_size)
        
        # Temperature-scaled cross entropy
        temperature = 0.07
        logits = scores / temperature
        loss = torch.nn.functional.cross_entropy(logits, labels)
        
        return loss
    
    def train_step(self, queries: List[str], passages: List[str]) -> float:
        """One training step.
        
        Args:
            queries: List of queries (positive passages are at same indices)
            passages: List of passages
        
        Returns:
            Loss value
        """
        self.optimizer.zero_grad()
        
        # Forward: queries[i] should match passages[i]
        scores = self.model(queries, passages)  # (B, B)
        loss = self.compute_contrastive_loss(scores)
        
        loss.backward()
        self.optimizer.step()
        
        return loss.item()

# Test bi-encoder training
print("\n=== Bi-Encoder Training ===")
trainer = DPRTrainer(embedding_dim=128)

# Simple QA dataset
queries_train = [
    "What is the capital of France?",
    "Where is the Eiffel Tower?",
    "How tall is Mount Fuji?",
]

passages_train = [
    "Paris is the capital of France.",
    "The Eiffel Tower is in Paris.",
    "Mount Fuji is 3,776 meters tall.",
]

# Train for a few steps
losses = []
for step in range(5):
    loss = trainer.train_step(queries_train, passages_train)
    losses.append(loss)
    if (step + 1) % 1 == 0:
        print(f"Step {step+1}: loss = {loss:.4f}")

print(f"\nFinal loss: {losses[-1]:.4f}")

## Real-World Example 1: FAISS Indexing for Large-Scale Retrieval

In [ ]:
class FAISSRetriever:
    """Dense retriever with FAISS indexing for scalable search."""
    
    def __init__(self, embedding_dim: int = 256):
        """Initialize FAISS retriever.
        
        Args:
            embedding_dim: Embedding dimension
        """
        self.embedding_dim = embedding_dim
        self.passages = []
        self.embeddings = None
        self.index = None
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for texts.
        
        Args:
            texts: List of text strings
        
        Returns:
            Embeddings (len(texts), embedding_dim)
        """
        embs = []
        for text in texts:
            np.random.seed(hash(text) % (2**31))
            emb = np.random.randn(self.embedding_dim).astype(np.float32)
            # L2 normalize
            emb = emb / (np.linalg.norm(emb) + 1e-8)
            embs.append(emb)
        
        return np.array(embs)
    
    def build_index_brute_force(self, embeddings: np.ndarray):
        """Build exact kNN index (brute force).
        
        Args:
            embeddings: (num_passages, embedding_dim)
        """
        # In production, use FAISS library
        # For demo, use simple brute-force search
        self.embeddings = embeddings
        print(f"Built brute-force index: {len(embeddings)} passages, {embeddings.shape[1]} dims")
    
    def index_passages(self, passages: List[str]):
        """Index passages for retrieval.
        
        Args:
            passages: List of passage texts
        """
        self.passages = passages
        embeddings = self.generate_embeddings(passages)
        self.build_index_brute_force(embeddings)
    
    def retrieve(self, query: str, top_k: int = 10) -> Tuple[List[str], List[float]]:
        """Retrieve top-K passages.
        
        Args:
            query: Query text
            top_k: Number of results
        
        Returns:
            (passages, scores)
        """
        q_emb = self.generate_embeddings([query])[0]  # (dim,)
        
        # Cosine similarity
        scores = np.dot(self.embeddings, q_emb)  # (num_passages,)
        
        # Top-K
        top_k_idx = np.argsort(scores)[-top_k:][::-1]
        
        retrieved = [self.passages[i] for i in top_k_idx]
        retrieved_scores = [float(scores[i]) for i in top_k_idx]
        
        return retrieved, retrieved_scores

# Test FAISS-style retriever on larger corpus
print("\n=== Large-Scale FAISS Retrieval ===")

faiss_retriever = FAISSRetriever(embedding_dim=256)

# Build larger corpus
large_corpus = [
    "Paris is the capital of France.",
    "The Eiffel Tower is in Paris and is 330 meters tall.",
    "France is a country in Western Europe with a population of 67 million.",
    "Tokyo is the capital and largest city of Japan.",
    "Japan is an island nation in East Asia with a population of 125 million.",
    "Mount Fuji is the highest mountain in Japan at 3,776 meters.",
    "Berlin is the capital of Germany.",
    "Germany is located in Central Europe.",
    "Rome is the capital of Italy and is known as the Eternal City.",
    "Italy is a Mediterranean country with a rich historical heritage.",
]

faiss_retriever.index_passages(large_corpus)

# Test retrieval
test_query = "What is the height of the famous tower in the capital of France?"
results, scores = faiss_retriever.retrieve(test_query, top_k=5)

print(f"Query: {test_query}")
print(f"\nTop-5 Retrieved Passages:")
for i, (passage, score) in enumerate(zip(results, scores), 1):
    print(f"  [{i}] (score={score:.3f}) {passage}")

## Real-World Example 2: Scaling Analysis - Retrieval Performance at Different Corpus Sizes

## Real-World Example 3: Retrieval Evaluation on QA Benchmarks

In [ ]:
class QAEvaluator:
    """Evaluate retrieval quality on QA tasks."""
    
    def __init__(self, retriever):
        self.retriever = retriever
    
    def evaluate_recall(self, qa_pairs: List[Dict], top_k_values: List[int]) -> Dict:
        """Evaluate retrieval recall at different K values.
        
        Args:
            qa_pairs: List of {"question": str, "answer_passage": str}
            top_k_values: List of K values to evaluate
        
        Returns:
            Recall@K for each K
        """
        results = {k: [] for k in top_k_values}
        
        for qa in qa_pairs:
            question = qa["question"]
            expected_passage = qa["answer_passage"]
            
            # Retrieve
            retrieved, scores = self.retriever.retrieve(question, top_k=max(top_k_values))
            
            # Check recall at each K
            for k in top_k_values:
                # Check if expected passage is in top-K
                found = any(expected_passage == p for p in retrieved[:k])
                results[k].append(1.0 if found else 0.0)
        
        # Average
        return {k: np.mean(v) for k, v in results.items()}
    
    def evaluate_multiple_queries(self, qa_data: List[Dict], top_k: int = 5) -> Dict:
        """Full evaluation on QA dataset.
        
        Args:
            qa_data: QA dataset
            top_k: Primary K value
        
        Returns:
            Detailed metrics
        """
        recall = self.evaluate_recall(qa_data, [1, 3, 5, 10])
        
        return {
            "recall_at_k": recall,
            "num_questions": len(qa_data),
        }

# Simulate QA evaluation
print("\n=== QA Evaluation ===")

# Synthetic QA dataset
qa_dataset = [
    {"question": "What is the capital of France?", "answer_passage": "Paris is the capital of France."},
    {"question": "How tall is the Eiffel Tower?", "answer_passage": "The Eiffel Tower is in Paris and is 330 meters tall."},
    {"question": "What is the population of Japan?", "answer_passage": "Japan is an island nation in East Asia with a population of 125 million."},
]

evaluator = QAEvaluator(faiss_retriever)
eval_results = evaluator.evaluate_multiple_queries(qa_dataset)

print(f"\nEvaluated on {eval_results['num_questions']} QA pairs")
print(f"\nRecall@K:")
for k, recall in eval_results['recall_at_k'].items():
    print(f"  Recall@{k:2d}: {recall:.1%}")

## Comparison: Dense vs Sparse Retrieval

In [ ]:
# Simulate accuracy comparison between sparse (BM25) and dense (neural) retrieval
# Based on realistic numbers from DPR paper and benchmarks

retriever_types = [\"Sparse (BM25)\", \"Dense (DPR)\", \"Hybrid (BM25+Dense)\"]\nrecall_at_k_values = {\n    \"Sparse (BM25)\": [0.62, 0.73, 0.78],        # Top-1, Top-5, Top-20\n    \"Dense (DPR)\": [0.71, 0.82, 0.88],          # Higher semantic matching\n    \"Hybrid (BM25+Dense)\": [0.74, 0.84, 0.90],  # Best of both\n}\nk_values = [1, 5, 20]\n\n# Speed comparison (ms per query)\nlatencies = {\n    \"Sparse (BM25)\": 5,\n    \"Dense (DPR)\": 50,\n    \"Hybrid (BM25+Dense)\": 55,\n}\n\n# Plot\nfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))\n\n# Recall comparison\nx_pos = np.arange(len(k_values))\nwidth = 0.25\n\nfor i, ret_type in enumerate(retriever_types):\n    recalls = recall_at_k_values[ret_type]\n    ax1.bar(x_pos + i*width, recalls, width, label=ret_type, alpha=0.8)\n\nax1.set_ylabel('Recall', fontsize=11)\nax1.set_xlabel('K (top-K retrieved)', fontsize=11)\nax1.set_title('Retrieval Recall: Sparse vs Dense', fontsize=12, fontweight='bold')\nax1.set_xticks(x_pos + width)\nax1.set_xticklabels([f'Top-{k}' for k in k_values])\nax1.legend(fontsize=10)\nax1.set_ylim([0, 1.0])\nax1.grid(axis='y', alpha=0.3)\n\n# Speed vs Accuracy trade-off\nfor ret_type in retriever_types:\n    recall_at_5 = recall_at_k_values[ret_type][1]  # Recall@5\n    latency = latencies[ret_type]\n    ax2.scatter(latency, recall_at_5, s=400, alpha=0.7, label=ret_type)\n\nax2.set_xlabel('Latency (ms per query)', fontsize=11)\nax2.set_ylabel('Recall@5', fontsize=11)\nax2.set_title('Speed vs Accuracy Trade-off', fontsize=12, fontweight='bold')\nax2.legend(fontsize=10)\nax2.grid(alpha=0.3)\nax2.set_xlim([0, 70])\nax2.set_ylim([0.65, 0.95])\n\nplt.tight_layout()\nplt.savefig('/tmp/neural_search_comparison.png', dpi=100, bbox_inches='tight')\nplt.show()\n\nprint(\"\\n=== Retriever Comparison ===\")\nprint(f\"{'Retriever':<20} | {'Recall@1':<10} | {'Recall@5':<10} | {'Latency (ms)':<12}\")\nprint(\"-\" * 60)\nfor ret_type in retriever_types:\n    recalls = recall_at_k_values[ret_type]\n    latency = latencies[ret_type]\n    print(f\"{ret_type:<20} | {recalls[0]:<10.1%} | {recalls[1]:<10.1%} | {latency:<12}\")

## Key Takeaways

**Core idea:**
Neural search uses learned dense embeddings to map queries and passages to a shared space, enabling semantic similarity-based retrieval. This overcomes the limitations of sparse keyword-based retrieval (BM25).

**Key mechanisms:**
1. **Bi-encoder architecture:** Separate query and passage encoders trained jointly
2. **Contrastive learning:** Pull relevant pairs close, push non-relevant pairs apart
3. **Vector indexing:** FAISS enables fast approximate nearest-neighbor search at scale
4. **Retrieval evaluation:** Measure recall@K (does gold passage appear in top-K?)

**Trade-offs:**
- **Accuracy vs Speed:** Dense retrieval (higher accuracy, slower); Sparse/BM25 (faster, lower accuracy); Hybrid (best of both)
- **Indexing cost:** Large corpus requires careful index choice (IVFADC for speed, HNSW for accuracy)
- **Training data:** More labeled QA data improves encoder quality; in-batch negatives are free, hard negatives help significantly

**When to use Neural Search:**
- Semantic mismatch queries: user rephrases differently than answer text
- Open-domain QA: large corpus, high accuracy required
- Embedding-based: text similarity, recommendation, multimodal search
- NOT for: keyword-exact matching, real-time on-the-fly indexing

**Common pitfalls:**
1. Domain shift: trained on one domain, deployed on another -> fine-tune on target
2. Index staleness: new documents not indexed -> incremental indexing or dual-index
3. Poor negatives: training negatives too easy -> use BM25 hard negatives
4. Embedding dimension too large: memory bloat -> use quantization or PCA

**Related concepts:**
- Retrieval-Augmented Generation (RAG): combine dense retrieval with generation
- Cross-encoders: re-rank top-K retrieved passages (more accurate but slower)
- Embeddings: dense representations are foundation of modern NLP

## Exercises: Try It Yourself

1. **Train with different negatives:** Compare training with random negatives vs. BM25 hard negatives. Measure impact on dev set recall.

2. **Evaluate on real queries:** Use the retriever on your own questions and passages. Does it retrieve relevant information?

3. **Scaling experiment:** Measure latency and memory as corpus grows (1K to 1M passages). At what size does brute-force become impractical?

4. **Index comparison:** Implement FAISS with different index types (brute-force, IVFADC, HNSW). Measure accuracy vs latency trade-off.

5. **Re-ranking study:** After retrieving top-20 with dense encoder, use a cross-encoder to re-rank to top-5. Does it improve final accuracy?